Given two datasets: one containing signup details (including start and stop times) and another containing transaction details (such as amounts), determine the most profitable location based on signup duration and transaction amounts.

- Solution 
- Calculate Signup Duration
- Calculate Average Transaction Amount 
- Combine Signup and Transaction Data
- Calculate Ratio
- Group and Sort Results

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, unix_timestamp, when, avg

In [0]:
# Initialize Spark session
spark = SparkSession.builder.appName("ProfitableLocation").getOrCreate()

In [0]:
signups_data = [
    (1, '2020-01-01 10:00:00', '2020-01-01 12:00:00', 101, 'New York'),
    (2, '2020-01-02 11:00:00', '2020-01-02 13:00:00', 102, 'Los Angeles'),
    (3, '2020-01-03 10:00:00', '2020-01-03 14:00:00', 103, 'Chicago'),
    (4, '2020-01-04 09:00:00', '2020-01-04 10:30:00', 101, 'San Francisco'),
    (5, '2020-01-05 08:00:00', '2020-01-05 11:00:00', 102, 'New York')
]
transactions_data = [
    (1, 1, '2020-01-01 10:30:00', 50.00),
    (2, 1, '2020-01-01 11:00:00', 30.00),
    (3, 2, '2020-01-02 11:30:00', 100.00),
    (4, 2, '2020-01-02 12:00:00', 75.00),
    (5, 3, '2020-01-03 10:30:00', 120.00),
    (6, 4, '2020-01-04 09:15:00', 80.00),
    (7, 5, '2020-01-05 08:30:00', 90.00)
]
# Define columns for signups DataFrame
signups_columns = [
  "signup_id", 
  "signup_start_date", 
  "signup_stop_date", 
  "plan_id", 
  "location"
]

In [0]:
# Define columns for transactions DataFrame
transactions_columns = [
  "transaction_id", 
  "signup_id", 
  "transaction_start_date", 
  "amt"
]

In [0]:
signups_df = spark.createDataFrame(signups_data, signups_columns)
transactions_df = spark.createDataFrame(transactions_data, transactions_columns)


In [0]:
# Calculate signup duration in minutes for each signup
signups_df = signups_df.withColumn(
    "signup_duration_minutes", 
    (unix_timestamp("signup_stop_date") - unix_timestamp("signup_start_date")) / 60
)

In [0]:
# Calculate average transaction amount for each signup
transaction_avg_df = transactions_df.groupBy("signup_id").agg(avg("amt").alias("avg_transaction_amount"))


In [0]:
# Join the signups with transaction averages
joined_df = signups_df.join(transaction_avg_df, on="signup_id", how="inner")


In [0]:
# Group by location and calculate average duration, average transaction amount, and ratio
result_df = joined_df.groupBy("location").agg(
    avg("signup_duration_minutes").alias("avg_duration"),
    avg("avg_transaction_amount").alias("avg_transaction_amount")
)

In [0]:
# Calculate ratio of transaction amount to signup duration
result_df = result_df.withColumn(
    "ratio", 
    when(col("avg_duration") != 0, col("avg_transaction_amount") / col("avg_duration")).otherwise(0)
)

In [0]:
# Sort by ratio from highest to lowest
result_df = result_df.orderBy(col("ratio"), ascending=False)


In [0]:
# Show the final result
result_df.show(truncate=False)

+-------------+------------+----------------------+-------------------+
|location     |avg_duration|avg_transaction_amount|ratio              |
+-------------+------------+----------------------+-------------------+
|San Francisco|90.0        |80.0                  |0.8888888888888888 |
|Los Angeles  |120.0       |87.5                  |0.7291666666666666 |
|Chicago      |240.0       |120.0                 |0.5                |
|New York     |150.0       |65.0                  |0.43333333333333335|
+-------------+------------+----------------------+-------------------+

